# Driven Sideband Quasienergy Scan


This notebook introduces `cqed_sim.floquet` through a transmon-storage sideband problem. We sweep a periodic drive across the red-sideband frequency, solve the Floquet problem at each point, and track the resulting quasienergy branches.

The physical question is where the drive-dressed spectrum shows the strongest avoided crossing. That minimum quasienergy gap is a compact way to locate the most resonant part of the periodic-drive sweep.


## Imports


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from tutorials.workflow_tutorial_support import configure_notebook_style

configure_notebook_style()

from cqed_sim import FloquetConfig, FloquetProblem, SidebandDriveSpec, build_target_drive_term, run_floquet_sweep
from cqed_sim.core import DispersiveTransmonCavityModel, FrameSpec
from tutorials.tutorial_support import GHz, MHz


## Physics / model definition


In [ ]:
model = DispersiveTransmonCavityModel(
    omega_c=GHz(5.05),
    omega_q=GHz(6.25),
    alpha=MHz(-250.0),
    chi=MHz(-15.0),
    kerr=0.0,
    n_cav=3,
    n_tr=3,
)
frame = FrameSpec()
sideband = SidebandDriveSpec(mode="storage", lower_level=0, upper_level=1, sideband="red")
center_frequency = model.sideband_transition_frequency(
    cavity_level=0,
    lower_level=0,
    upper_level=1,
    sideband="red",
    frame=frame,
)
scan_detuning_mhz = np.linspace(-0.25, 0.25, 61)
drive_frequencies = center_frequency + 2.0 * np.pi * 1.0e6 * scan_detuning_mhz


## Pulse / sequence construction


In [ ]:
problems = []
for frequency in drive_frequencies:
    drive = build_target_drive_term(
        model,
        sideband,
        amplitude=MHz(0.03),
        frequency=float(frequency),
        waveform="cos",
        label="effective_red_sideband_drive",
    )
    problems.append(
        FloquetProblem(
            model=model,
            frame=frame,
            periodic_terms=(drive,),
            period=2.0 * np.pi / float(frequency),
            label="transmon_storage_sideband_scan",
        )
    )

print("Reference red-sideband frequency [GHz]:", center_frequency / (2.0 * np.pi * 1.0e9))


## Simulation


In [ ]:
sweep = run_floquet_sweep(
    problems,
    parameter_values=scan_detuning_mhz,
    config=FloquetConfig(n_time_samples=96),
)

tracked_quasienergies_mhz = sweep.tracked_quasienergies / (2.0 * np.pi * 1.0e6)
minimum_gap_mhz = np.min(np.abs(np.diff(tracked_quasienergies_mhz, axis=1)), axis=1)
best_index = int(np.argmin(minimum_gap_mhz))

print("Minimum tracked gap [MHz]:", float(minimum_gap_mhz[best_index]))
print("Detuning at minimum gap [MHz]:", float(scan_detuning_mhz[best_index]))


## Analysis / visualization


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9.0, 7.0), sharex=True)

for branch in range(tracked_quasienergies_mhz.shape[1]):
    axes[0].plot(scan_detuning_mhz, tracked_quasienergies_mhz[:, branch], linewidth=1.2)
axes[0].axvline(scan_detuning_mhz[best_index], color="black", linestyle="--", linewidth=1.0)
axes[0].set_ylabel("Tracked quasienergy / 2pi (MHz)")
axes[0].set_title("Sideband Floquet branch tracking")
axes[0].grid(True, alpha=0.3)

axes[1].plot(scan_detuning_mhz, minimum_gap_mhz, color="#E45756", linewidth=1.6)
axes[1].axvline(scan_detuning_mhz[best_index], color="black", linestyle="--", linewidth=1.0)
axes[1].set_xlabel("Drive detuning from red sideband (MHz)")
axes[1].set_ylabel("Minimum adjacent branch gap (MHz)")
axes[1].set_title("Avoided-crossing diagnostic")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Interpretation


The sweep tracks the same dressed branches continuously rather than re-sorting them independently at each parameter point. That is why the minimum-gap curve is physically interpretable: it points to the drive setting where the periodic sideband drive hybridizes the states most strongly.

This is one of the main advantages of a Floquet workflow over repeated static diagonalization. The periodic drive becomes part of the eigenproblem rather than a small perturbation added afterward.


## Variations / exercises


- Increase the drive amplitude to see the avoided crossing widen.
- Expand the cavity or transmon truncation to check whether the tracked branches are converged.
- Use the next Floquet notebook to interpret branch structure in terms of multiphoton resonance conditions.
